In [1]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm


In [2]:
anime_to_consider = pd.read_csv("AnimeWithImages.csv") 
images_folder = "Anime_images" 
size_to = (75,75)

In [3]:
import cv2
images = np.ndarray((anime_to_consider.shape[0],size_to[1]*size_to[0]*3))
for i,(ind,row) in tqdm(enumerate(anime_to_consider.iterrows()),total=anime_to_consider.shape[0]):
    filename=f"{images_folder}/anime_{row["anime_id"]}_image.jpg"
    img = cv2.imread(filename)
    assert img is not None, filename
    img = cv2.resize(img,size_to)
    images[i]=img.flatten()


100%|██████████| 2000/2000 [00:05<00:00, 368.46it/s]


In [ ]:
svd_results_path = "SVDResults"
os.makedirs(svd_results_path,exist_ok=True)
means = images.mean(axis=0)
np.save(f"{svd_results_path}/means.npy",means)
images = images- means
np.save(f"{svd_results_path}/images.npy",images)
u, s, v = np.linalg.svd(images)

In [5]:
c = np.zeros((u.shape[0],v.shape[0]))
r= np.arange(s.shape[0])
c[r,r]=s[r]
image_reconstruction_attempt = u @ c @ v
assert image_reconstruction_attempt.shape == images.shape
diff = image_reconstruction_attempt-images
print(np.sum(diff**2))

2.786304800767815e-18


In [6]:

np.save(f"{svd_results_path}/U_svd_matrix.npy",u)
np.save(f"{svd_results_path}/S_svd_matrix.npy",s)
np.save(f"{svd_results_path}/V_svd_matrix.npy",v)

In [7]:
import random
feature_amount = 20
images = np.load("SVDResults/images.npy")
images_folder = "Anime_images"
ind = random.randint(0, anime_to_consider.shape[0])
row = anime_to_consider.iloc[ind]
means = np.load("SVDResults/means.npy")
c = np.zeros((u.shape[0], v.shape[0]))
r = np.arange(s.shape[0])
c[r, r] = s[r]
image_reconstruction_attempt = u[:, :feature_amount] @ np.diag(s[:feature_amount]) @ v[:feature_amount]
img = image_reconstruction_attempt[ind] + means
filename = f"{images_folder}/anime_{row['anime_id']}_image.jpg"
image = cv2.imread(filename)
image = cv2.resize(image, size_to)
img = img.reshape(size_to[1], size_to[0], 3)
diff = image - img
cv2.imwrite("image-real.jpg", image)
cv2.imwrite("image-recon.jpg", img)
print(s[:feature_amount].sum() / s.sum())

522022.40005577373
0.09450624201540173
